In [2]:
import duckdb
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import os
from pathlib import Path
import geopandas as gpd
import xml.etree.ElementTree as ET
import re
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [3]:
con = duckdb.connect()
con.sql("INSTALL ducklake; LOAD ducklake;")
con.sql("INSTALL spatial; LOAD spatial;")

In [32]:
con.sql(f"""
            ATTACH 'ducklake:mobility.ducklake' AS my_ducklake;
            USE my_ducklake;
            CREATE SCHEMA IF NOT EXISTS silver;
                """)

BinderException: Binder Error: Failed to attach DuckLake MetaData "__ducklake_metadata_my_ducklake" at path + "mobility.ducklake"Unique file handle conflict: Cannot attach "__ducklake_metadata_my_ducklake" - the database file "mobility.ducklake" is already attached by database "__ducklake_metadata_my_ducklake"

In [33]:
def _ensure_gold_schema(con: duckdb.DuckDBPyConnection) -> None:
    con.sql("CREATE SCHEMA IF NOT EXISTS gold;")

In [31]:
def _get_zone_types(con, year: int) -> list[str]:
    rows = con.sql(
        f"""
        SELECT DISTINCT zone_type
        FROM silver.od_trips
        WHERE YEAR(date) = {year}
        ORDER BY 1
        """
    ).fetchall()
    return [r[0] for r in rows]

In [7]:
_get_zone_types(con, 2023)

['distritcs', 'gaus']

In [30]:
def _label_clusters(df_days: pd.DataFrame) -> dict[int, str]:
    df_days = df_days.copy()
    df_days["is_weekday"] = df_days["weekday"] <= 4

    stats = (
        df_days.groupby("cluster_id")
        .agg(
            weekday_rate=("is_weekday", "mean"),
            avg_total_trips=("total_trips", "mean"),
            avg_morning=("morning_share", "mean"),
            avg_evening=("evening_share", "mean"),
            n_days=("trip_date", "count"),
        )
        .sort_values(["weekday_rate", "avg_total_trips"], ascending=[True, True])
    )

    cids = list(stats.index)
    label_map: dict[int, str] = {}

    if len(cids) == 1:
        label_map[cids[0]] = "Single pattern"
        return label_map

    if len(cids) == 2:
        label_map[cids[0]] = "Weekend"
        label_map[cids[1]] = "Weekday"
        return label_map

    label_map[cids[0]] = "Weekend"
    label_map[cids[-1]] = "Weekday"
    for cid in cids[1:-1]:
        label_map[cid] = "Holiday"

    return label_map

#CLUSTERS: k-means
def build_day_clusters(
    con,
    year: int,
    n_clusters: int = 3,
    zone_types: list[str] | None = None,
    include_volume_in_clustering: bool = True,
    random_state: int = 42,
) -> None:
    _ensure_gold_schema(con)

    if zone_types is None:
        zone_types = _get_zone_types(con, year)

    if not zone_types:
        print(f"[BQ1] No zone_types found for year={year}.")
        return

    zone_list_sql = ", ".join([f"'{z}'" for z in zone_types])

    df_temporal = con.sql(
        f"""
        SELECT
            CAST(date AS DATE) AS trip_date,
            zone_type,
            EXTRACT(HOUR FROM date) AS hour_of_day,
            SUM(n_trips) AS total_trips
        FROM silver.od_trips
        WHERE YEAR(date) = {year}
          AND zone_type IN ({zone_list_sql})
        GROUP BY 1,2,3
        ORDER BY 1,2,3
        """
    ).df()
    #print(df_temporal)
    if df_temporal.empty:
        print(f"[BQ1] No data in silver.od_trips for year={year}.")
        return

    pivot = (
        df_temporal.pivot_table(
            index=["trip_date", "zone_type"],
            columns="hour_of_day",
            values="total_trips",
            aggfunc="sum",
            fill_value=0.0,
        )
        .sort_index()
    )
    pivot = pivot.reindex(columns=list(range(24)), fill_value=0.0)
    print(pivot)
    daily_total = pivot.sum(axis=1)
    shares = pivot.div(daily_total.replace(0, np.nan), axis=0).fillna(0.0)

    peak_hour = pivot.idxmax(axis=1).astype(int)
    
    morning_share = shares.loc[:, 7:10].sum(axis=1)
    evening_share = shares.loc[:, 17:20].sum(axis=1)

    df_features = shares.copy()
    df_features.columns = [f"share_h{h:02d}" for h in df_features.columns]
    df_features = df_features.reset_index()

    df_features["total_trips"] = daily_total.values
    df_features["peak_hour"] = peak_hour.values
    df_features["morning_share"] = morning_share.values
    df_features["evening_share"] = evening_share.values
    df_features["weekday"] = pd.to_datetime(df_features["trip_date"]).dt.weekday
    #print(df_features)
    share_cols = [c for c in df_features.columns if c.startswith("share_h")]

    out_rows: list[pd.DataFrame] = []

    for zt in zone_types:
        df_zt = df_features[df_features["zone_type"] == zt].copy()
        if df_zt.empty:
            continue

        n_days = len(df_zt)
        k = min(n_clusters, n_days)
        if k < 2:
            df_zt["cluster_id"] = 0
            df_zt["pattern_name"] = "Single pattern (insufficient days)"
            out_rows.append(df_zt[["trip_date", "zone_type", "cluster_id", "pattern_name"]])
            continue

        X_parts = [df_zt[share_cols].to_numpy(dtype=float)]
        if include_volume_in_clustering:
            vol = np.log1p(df_zt["total_trips"].to_numpy(dtype=float)).reshape(-1, 1)
            pk = df_zt["peak_hour"].to_numpy(dtype=float).reshape(-1, 1)
            X_parts += [vol, pk]

        X = np.hstack(X_parts)
        X_scaled = StandardScaler().fit_transform(X)

        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        df_zt["cluster_id"] = km.fit_predict(X_scaled)

        label_map = _label_clusters(df_zt)
        df_zt["pattern_name"] = df_zt["cluster_id"].map(label_map).fillna(
            "Pattern " + df_zt["cluster_id"].astype(str)
        )

        out_rows.append(df_zt[["trip_date", "zone_type", "cluster_id", "pattern_name"]])
        print(f"[BQ1] zone_type={zt}: {n_days} days clustered into k={k}")

    df_clusters = pd.concat(out_rows, ignore_index=True)
    df_clusters["ingestion_date"] = pd.Timestamp.utcnow()

    con.register("df_day_clusters", df_clusters)
    con.sql("CREATE OR REPLACE TABLE gold.day_clusters AS SELECT * FROM df_day_clusters;")
    con.unregister("df_day_clusters")

    print("[BQ1] gold.day_clusters created. Days per cluster:")
    con.sql(
        """
        SELECT zone_type, cluster_id, pattern_name, COUNT(*) AS n_days
        FROM gold.day_clusters
        GROUP BY 1,2,3
        ORDER BY zone_type, n_days DESC
        """
    ).show()
    return df_clusters

In [ ]:
df_clusters = build_day_clusters(
    con,
    2023,
    
    zone_types = ['distritcs', 'gaus'],
    
)

hour_of_day                  0          1          2          3          4   \
trip_date  zone_type                                                          
2023-06-01 distritcs  3051038.8  1599592.7  1111146.3   925599.0  1026548.5   
           gaus       3050825.6  1599429.2  1111037.9   925469.3  1026434.3   
2023-06-02 distritcs  3032867.6  1663452.9  1148014.4   944743.3  1066247.9   
           gaus       3032791.4  1663409.2  1148003.6   944723.5  1066216.3   
2023-06-03 distritcs  4129979.8  2468426.4  1616878.5  1254581.5  1177541.0   
           gaus       4130043.8  2468442.5  1616935.5  1254634.4  1177563.2   
2023-06-04 distritcs  4379115.4  2788761.6  1875564.9  1430211.5  1281781.7   
           gaus       4378893.3  2788615.3  1875481.7  1430111.2  1281692.7   
2023-06-05 distritcs  2962613.5  1622725.1  1171655.2   946556.1  1039930.2   
           gaus       2962522.4  1622675.5  1171616.0   946529.8  1039927.7   
2023-06-06 distritcs  2706354.9  1512096.9  1100338.

In [21]:
df_clusters

,trip_date,zone_type,cluster_id,pattern_name,ingestion_date
0,2023-06-01,distritcs,0,Weekday,2026-01-03 15:21:41.799546+00:00
1,2023-06-02,distritcs,0,Weekday,2026-01-03 15:21:41.799546+00:00
2,2023-06-03,distritcs,2,Holiday,2026-01-03 15:21:41.799546+00:00
3,2023-06-04,distritcs,1,Weekend,2026-01-03 15:21:41.799546+00:00
4,2023-06-05,distritcs,0,Weekday,2026-01-03 15:21:41.799546+00:00
5,2023-06-06,distritcs,0,Weekday,2026-01-03 15:21:41.799546+00:00
6,2023-06-07,distritcs,0,Weekday,2026-01-03 15:21:41.799546+00:00
7,2023-06-01,gaus,0,Weekday,2026-01-03 15:21:41.799546+00:00
8,2023-06-02,gaus,0,Weekday,2026-01-03 15:21:41.799546+00:00
9,2023-06-03,gaus,2,Holiday,2026-01-03 15:21:41.799546+00:00


In [12]:
con.sql("SELECT * FROM silver.od_trips")

┌─────────────────────┬───────────┬───────────┬────────────────┬─────────────────┬──────────────────────┬───────────────────┬────────────────────┬────────────┬───────────┬───────────┬─────────┬───────────────────────┬─────────────────────┬──────────────────────────┬────────────────────────────┐
│        date         │ zone_type │ id_origin │ id_destination │ origin_activity │ destination_activity │ distance_group_km │ residence_province │ rent_group │ age_group │ sex_group │ n_trips │ trips_total_length_km │ origin_activity_std │ destination_activity_std │       ingestion_date       │
│      timestamp      │  varchar  │  varchar  │    varchar     │     varchar     │       varchar        │      varchar      │      varchar       │  varchar   │  varchar  │  varchar  │ double  │        double         │       boolean       │         boolean          │         timestamp          │
├─────────────────────┼───────────┼───────────┼────────────────┼─────────────────┼──────────────────────┼───────

In [11]:
year = 2023
zone_types = ['distritcs', 'gaus']
zone_list_sql = ", ".join([f"'{z}'" for z in zone_types])
con.sql(f"""SELECT
            CAST(t.date AS DATE) AS trip_date,
            t.zone_type,
            EXTRACT(HOUR FROM t.date) AS hour_of_day,

            TRY_CAST(t.id_origin AS INTEGER) AS id_origin,
            TRY_CAST(t.id_destination AS INTEGER) AS id_destination,
            t.distance_group_km,
            SUM(t.n_trips) AS daily_trips,
            SUM(t.trips_total_length_km) AS daily_total_length_km,
            
            SUM(t.trips_total_length_km) / NULLIF(SUM(t.n_trips), 0) AS daily_avg_trip_length_km
        FROM silver.od_trips t
        WHERE YEAR(t.date) = {year}
          AND t.zone_type IN ({zone_list_sql})
        GROUP BY 1,2,3,4,5,6
        """)

┌────────────┬───────────┬─────────────┬───────────┬────────────────┬───────────────────┬────────────────────┬───────────────────────┬──────────────────────────┐
│ trip_date  │ zone_type │ hour_of_day │ id_origin │ id_destination │ distance_group_km │    daily_trips     │ daily_total_length_km │ daily_avg_trip_length_km │
│    date    │  varchar  │    int64    │   int32   │     int32      │      varchar      │       double       │        double         │          double          │
├────────────┼───────────┼─────────────┼───────────┼────────────────┼───────────────────┼────────────────────┼───────────────────────┼──────────────────────────┤
│ 2023-06-07 │ distritcs │          13 │     10920 │          11409 │ 2-10              │ 108.80000000000001 │    371.41599999999994 │        3.413749999999999 │
│ 2023-06-07 │ distritcs │          17 │     10920 │          11409 │ 2-10              │              145.7 │     543.7520000000001 │        3.731997254632808 │
│ 2023-06-07 │ distritcs │  

In [28]:
def build_typical_day_patterns(
    con,
    year: int = 2023,
    zone_types: list[str] | None = None,
) -> None:
    _ensure_gold_schema(con)

    if zone_types is None:
        zone_types = _get_zone_types(con, year)

    if not zone_types:
        print(f"[BQ1] No zone_types found for year={year}.")
        return

    zone_list_sql = ", ".join([f"'{z}'" for z in zone_types])

    con.sql(
        f"""
        CREATE OR REPLACE TEMP TABLE _daily_od_hour AS
        SELECT
            CAST(t.date AS DATE) AS trip_date,
            t.zone_type,
            EXTRACT(HOUR FROM t.date) AS hour_of_day,

            TRY_CAST(t.id_origin AS INTEGER) AS id_origin,
            TRY_CAST(t.id_destination AS INTEGER) AS id_destination,
            t.distance_group_km,
            SUM(t.n_trips) AS daily_trips,
            SUM(t.trips_total_length_km) AS daily_total_length_km,
            
            SUM(t.trips_total_length_km) / NULLIF(SUM(t.n_trips), 0) AS daily_avg_trip_length_km
        FROM silver.od_trips t
        WHERE YEAR(t.date) = {year}
          AND t.zone_type IN ({zone_list_sql})
        GROUP BY 1,2,3,4,5,6
        """
    )

    con.sql("DELETE FROM _daily_od_hour WHERE id_origin IS NULL OR id_destination IS NULL;")

    con.sql(
        """
        CREATE OR REPLACE TABLE gold.typical_day_patterns AS
        SELECT
            dc.zone_type,
            dc.cluster_id,
            dc.pattern_name,
            d.hour_of_day,
            d.id_origin,
            d.id_destination,

            ROUND(AVG(d.daily_trips),2) AS avg_trips_per_day,
            ROUND(AVG(d.daily_avg_trip_length_km),2) AS avg_trip_length_km,
            d.distance_group_km as distance_group_km,
            ROUND(SUM(d.daily_trips),2) AS total_trips_in_cluster_sample,
            COUNT(DISTINCT d.trip_date) AS n_days_in_cluster,

            CURRENT_TIMESTAMP AS ingestion_date
        FROM _daily_od_hour d
        JOIN gold.day_clusters dc
          ON d.trip_date = dc.trip_date
         AND d.zone_type = dc.zone_type
        GROUP BY
            dc.zone_type, dc.cluster_id, dc.pattern_name,
            d.hour_of_day, d.id_origin, d.id_destination, distance_group_km
        """
    )

    print("[BQ1] gold.typical_day_patterns created. Rows per pattern:")
    con.sql(
        """
        SELECT zone_type, pattern_name, COUNT(*) AS rows
        FROM gold.typical_day_patterns
        GROUP BY 1,2
        ORDER BY zone_type, rows DESC
        """
    ).show()

In [16]:
con.sql("""
    CREATE OR REPLACE TABLE _staging_monthly_patterns (
        zone_type VARCHAR,
        cluster_id INTEGER,
        pattern_name VARCHAR,
        hour_of_day INTEGER,
        id_origin INTEGER,
        id_destination INTEGER,
        distance_group_km VARCHAR,
        
        -- Métricas acumulativas (Numeradores y Denominadores)
        sum_daily_trips DOUBLE,        -- Suma total de viajes
        sum_daily_length_km DOUBLE,    -- Suma total de km
        sum_daily_avg_len DOUBLE,      -- Suma de los promedios diarios (para replicar tu lógica original)
        days_count INTEGER             -- En cuántos días ocurrió este patrón
    )
""")
year = 2023
for month in range(1, 13):
    print(f"Procesando mes: {month} del año {year}...")
    
    con.sql(f"""
        INSERT INTO _staging_monthly_patterns
        WITH daily_stats AS (
            -- Paso A: Pre-agregación diaria (solo para este mes)
            -- Equivalente a tu tabla _daily_od_hour pero filtrada
            SELECT 
                CAST(t.date AS DATE) AS trip_date,
                t.zone_type,
                EXTRACT(HOUR FROM t.date) AS hour_of_day,
                TRY_CAST(t.id_origin AS INTEGER) AS id_origin,
                TRY_CAST(t.id_destination AS INTEGER) AS id_destination,
                t.distance_group_km,
                
                SUM(t.n_trips) AS daily_trips,
                SUM(t.trips_total_length_km) AS daily_total_length_km,
                -- Calculamos el promedio del día específico
                SUM(t.trips_total_length_km) / NULLIF(SUM(t.n_trips), 0) AS daily_avg_trip_length_km
            FROM silver.od_trips t
            WHERE YEAR(t.date) = {year} 
              AND MONTH(t.date) = {month}  -- <--- FILTRO CLAVE
              AND t.zone_type = 'distritcs'
            GROUP BY 1,2,3,4,5,6
            HAVING id_origin IS NOT NULL AND id_destination IS NOT NULL
        )
        -- Paso B: Unir con Clusters y colapsar a nivel mensual
        SELECT 
            ds.zone_type,
            dc.cluster_id,
            dc.pattern_name,
            ds.hour_of_day,
            ds.id_origin,
            ds.id_destination,
            ds.distance_group_km,
            
            -- Acumulamos para el mes
            SUM(ds.daily_trips) AS sum_daily_trips,
            SUM(ds.daily_total_length_km) AS sum_daily_length_km,
            SUM(ds.daily_avg_trip_length_km) AS sum_daily_avg_len,
            COUNT(DISTINCT ds.trip_date) AS days_count
            
        FROM daily_stats ds
        JOIN gold.day_clusters dc 
          ON ds.trip_date = dc.trip_date 
          AND ds.zone_type = dc.zone_type
        GROUP BY 1,2,3,4,5,6,7
    """)

con.sql("""
    CREATE OR REPLACE TABLE gold.typical_day_patterns AS
    SELECT 
        zone_type,
        cluster_id,
        pattern_name,
        hour_of_day,
        id_origin,
        id_destination,
        
        -- Cálculo final de promedios usando los acumulados
        ROUND(SUM(sum_daily_trips) / SUM(days_count), 2) AS avg_trips_per_day,
        
        -- Para replicar AVG(daily_avg_trip_length_km) de tu query original:
        -- (Suma de promedios diarios) / (Total de días)
        ROUND(SUM(sum_daily_avg_len) / SUM(days_count), 2) AS avg_trip_length_km,
        
        distance_group_km,
        
        ROUND(SUM(sum_daily_trips), 2) AS total_trips_in_cluster_sample,
        SUM(days_count) AS n_days_in_cluster,
        
        CURRENT_TIMESTAMP AS ingestion_date
    FROM _staging_monthly_patterns
    GROUP BY 
        zone_type, cluster_id, pattern_name, 
        hour_of_day, id_origin, id_destination, distance_group_km
""")

Procesando mes: 1 del año 2023...
Procesando mes: 2 del año 2023...
Procesando mes: 3 del año 2023...
Procesando mes: 4 del año 2023...
Procesando mes: 5 del año 2023...
Procesando mes: 6 del año 2023...
Procesando mes: 7 del año 2023...
Procesando mes: 8 del año 2023...
Procesando mes: 9 del año 2023...
Procesando mes: 10 del año 2023...
Procesando mes: 11 del año 2023...
Procesando mes: 12 del año 2023...


In [35]:
run_bq1(con,2023,3)

hour_of_day                  0          1          2          3          4   \
trip_date  zone_type                                                          
2023-06-01 distritcs  3051038.8  1599592.7  1111146.3   925599.0  1026548.5   
           gaus       3050825.6  1599429.2  1111037.9   925469.3  1026434.3   
2023-06-02 distritcs  3032867.6  1663452.9  1148014.4   944743.3  1066247.9   
           gaus       3032791.4  1663409.2  1148003.6   944723.5  1066216.3   
2023-06-03 distritcs  4129979.8  2468426.4  1616878.5  1254581.5  1177541.0   
           gaus       4130043.8  2468442.5  1616935.5  1254634.4  1177563.2   
2023-06-04 distritcs  4379115.4  2788761.6  1875564.9  1430211.5  1281781.7   
           gaus       4378893.3  2788615.3  1875481.7  1430111.2  1281692.7   
2023-06-05 distritcs  2962613.5  1622725.1  1171655.2   946556.1  1039930.2   
           gaus       2962522.4  1622675.5  1171616.0   946529.8  1039927.7   
2023-06-06 distritcs  2706354.9  1512096.9  1100338.

In [22]:
con.sql("""SELECT count(*)
FROM silver.od_trips
WHERE id_origin = id_destination""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     37405101 │
└──────────────┘

In [23]:
con.sql("SELECT count(*) FROM gold.typical_day_patterns")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     15805749 │
└──────────────┘

In [37]:
con.sql("SELECT count(*) FROM gold.typical_day_patterns WHERE zone_type = 'distritcs'")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     15805749 │
└──────────────┘

In [34]:

def run_bq1(
    con: duckdb.DuckDBPyConnection,
    year: int = 2023,
    n_clusters: int = 3,
    zone_types: list[str] | None = None,
    include_volume_in_clustering: bool = True,
    random_state: int = 42,
) -> None:
    build_day_clusters(
        con=con,
        year=year,
        n_clusters=n_clusters,
        zone_types=zone_types,
        include_volume_in_clustering=include_volume_in_clustering,
        random_state=random_state,
    )
    build_typical_day_patterns(
        con=con,
        year=year,
        zone_types=zone_types,
    )

In [46]:
con.sql("SELECT * FROM gold.typical_day_patterns ORDER BY avg_trips_per_day DESC")

┌───────────┬────────────┬──────────────┬─────────────┬───────────┬────────────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────────────────┬───────────────────┬───────────────────────────────┐
│ zone_type │ cluster_id │ pattern_name │ hour_of_day │ id_origin │ id_destination │ avg_trips_per_day │ avg_trip_length_km │ distance_group_km │ total_trips_in_cluster_sample │ n_days_in_cluster │        ingestion_date         │
│  varchar  │   int32    │   varchar    │    int64    │   int32   │     int32      │      double       │       double       │      varchar      │            double             │       int64       │   timestamp with time zone    │
├───────────┼────────────┼──────────────┼─────────────┼───────────┼────────────────┼───────────────────┼────────────────────┼───────────────────┼───────────────────────────────┼───────────────────┼───────────────────────────────┤
│ gaus      │          0 │ Weekday      │          14 │     63681 │          636

# Bussines cuestion 2

In [1]:
import duckdb
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import os
from pathlib import Path
import geopandas as gpd
import xml.etree.ElementTree as ET
import re
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [2]:
con = duckdb.connect()
con.sql("INSTALL ducklake; LOAD ducklake;")
con.sql("INSTALL spatial; LOAD spatial;")

In [3]:
con.sql(f"""
            ATTACH 'ducklake:mobility.ducklake' AS my_ducklake;
            USE my_ducklake;
            CREATE SCHEMA IF NOT EXISTS silver;
                """)

In [20]:
con.sql("SELECT  count(*) origen FROM bronze.gaus_info")

┌────────┐
│ origen │
│ int64  │
├────────┤
│   2203 │
└────────┘

In [18]:
con.sql("SELECT  DISTINCT origen FROM bronze.trips WHERE zone_type = 'GAU'")

┌────────────┐
│   origen   │
│  varchar   │
├────────────┤
│ GAU Ceuta  │
│ GAU Zamora │
│ PT112      │
│ 01047_AM   │
│ 01058_AM   │
│ 02012_AM   │
│ 02030      │
│ 03024      │
│ 03048      │
│ 03058      │
│   ·        │
│   ·        │
│   ·        │
│ 27010      │
│ 28038      │
│ 28093      │
│ 29017      │
│ 29039      │
│ 29041      │
│ 30003      │
│ 46031      │
│ FRJ22      │
│ FRK12      │
├────────────┤
│ 2149 rows  │
│ (20 shown) │
└────────────┘

In [55]:
con.sql("""SELECT count(*) FROM bronze.trips as br
        WHERE 
            YEAR(try_strptime(fecha::VARCHAR , '%Y%m%d')) = 2023
            AND TRIM(br.zone_type) = 'GAU'
            AND MONTH(try_strptime(fecha::VARCHAR, '%Y%m%d')) IN (6)
            AND DAY(try_strptime(fecha::VARCHAR, '%Y%m%d')) IN (1,)
        
        
        """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      7433590 │
└──────────────┘

In [257]:
con.sql("""--sql
        CREATE TABLE IF NOT EXISTS gold.gravity_pair_features (
            zone_type VARCHAR,
            year INTEGER,
            id_origin INTEGER,
            id_destination INTEGER,
            distance_km DOUBLE,
            actual_trips DOUBLE,
            pop_origin DOUBLE,
            inc_destination DOUBLE,
            x_ij DOUBLE,
            ingestion_date TIMESTAMP
        );""")

In [15]:
def _normalize_zone_type(zone_type: str) -> str:
    z = zone_type.strip().lower()
    aliases = {"distritos": "districts", "municipios": "municiples", "gau": "gaus"}
    z = aliases.get(z, z)

    valid = {"districts", "municiples", "gaus"}
    if z not in valid:
        raise ValueError(f"zone_type must be one of {sorted(valid)} (got: {zone_type})")
    return z
def _ensure_gold_schema(con: duckdb.DuckDBPyConnection) -> None:
    con.sql("CREATE SCHEMA IF NOT EXISTS gold;")

In [14]:
def _zone_types_sql(zone_types: list[str]) -> str:
    return ", ".join([f"'{_normalize_zone_type(z)}'" for z in zone_types])


In [ ]:
def build_gravity_pair_features(
    con: duckdb.DuckDBPyConnection,
    year: int = 2023,
    zone_type: str = "districts",
    dist_floor_km: float = 0.1,
    dist_power: int = 2,
    keep_only_with_distance: bool = True,
) -> None:
    _ensure_gold_schema(con)
    zt = _normalize_zone_type(zone_type)

    con.sql("""--sql
        CREATE TABLE IF NOT EXISTS gold.gravity_pair_features (
            zone_type VARCHAR,
            year INTEGER,
            id_origin INTEGER,
            id_destination INTEGER,
            distance_km DOUBLE,
            actual_trips DOUBLE,
            pop_origin DOUBLE,
            inc_destination DOUBLE,
            x_ij DOUBLE,
            ingestion_date TIMESTAMP
        );
    """)
    #x_ij= (P_i * E_j) / d^power, es el "score gravitatorio", es una proporción, si nos fijamos en la fórmula del documento
    #todavía falta un parámetro k para poder calcular los viajes teóricos


    query = f"""--sql
   
        WITH
        od AS (SELECT
                TRY_CAST(id_origin AS INTEGER) AS id_origin,
                TRY_CAST(id_destination AS INTEGER) AS id_destination,
                round(SUM(n_trips),1) AS actual_trips
            FROM silver.od_trips
            WHERE zone_type = 'distritcs'
              AND YEAR(date) = {year}
              AND id_origin <> id_destination
            GROUP BY 1,2 ),
       
        dist AS (
          SELECT
                zone_type,
                id_origin AS a,
                id_destination AS b,
                distance_km
            FROM silver.zone_pairs
            WHERE zone_type = 'districts'),
        aux_inc AS(
          SELECT DISTINCT id_districts_ine,
                id_{zt}_mitma
          FROM silver.ine_mitma_zones
          ),
        inc AS (
            SELECT 
              a.id_districts_mitma as id_zone,
              SUM(rent) AS inc
            FROM silver.average_rent i
            LEFT JOIN aux_inc a 
            ON i.id_zone = a.id_districts_ine 
            WHERE i.year = {year}
            AND id_{zt}_mitma NOT NULL
            GROUP BY id_{zt}_mitma
          ),
        aux_pop AS(
          SELECT DISTINCT id_sections_ine,
                id_{zt}_mitma
          FROM silver.ine_mitma_zones
        ),

      pop AS (
        SELECT id_districts_mitma as id_zone, 
          SUM(population) as population
        FROM silver.spain_population p 
        LEFT JOIN aux_pop a 
        ON p.id_zone = a.id_sections_ine 
        WHERE year = {year}
        AND id_districts_mitma NOT NULL
        GROUP BY id_{zt}_mitma
        ),
      joined AS(
        SELECT
                '{zt}' AS zone_type,
                {year} AS year,
                o.id_origin,
                o.id_destination,
                d.distance_km,
                o.actual_trips,
                po.population AS pop_origin,
                rd.inc AS inc_destination
            FROM od o
            LEFT JOIN dist d
              ON LEAST(o.id_origin, o.id_destination) = d.a
             AND GREATEST(o.id_origin, o.id_destination) = d.b
            LEFT JOIN pop po ON o.id_origin = po.id_zone
            LEFT JOIN inc rd ON o.id_destination = rd.id_zone)
      SELECT

            zone_type,
            year,
            id_origin,
            id_destination,
            distance_km,
            actual_trips,
            pop_origin,
            inc_destination,
            ROUND((pop_origin * inc_destination)/ NULLIF(POWER(GREATEST(distance_km, {0.1}), {2}), 0),3) AS x_ij,
            CURRENT_TIMESTAMP AS ingestion_date
        FROM joined

            """

    con.sql(f""" MERGE INTO gold.gravity_pair_features as target
            USING ({query} )AS source
                        ON target.zone_type = source.zone_type 
                        AND target.year = source.year
                        AND target.id_origin = source.id_origin
                        WHEN NOT MATCHED THEN
                            INSERT BY NAME;
            
            """)

    print(f"[BQ2] gold.gravity_pair_features built for zone_type={zt}, year={year}.")
    con.sql(
        f"""
        SELECT
          zone_type, year,
          COUNT(*) AS n_pairs,
          ROUND(SUM(actual_trips),2) AS total_actual_trips,
          ROUND(AVG(distance_km),2) AS avg_distance_km
        FROM gold.gravity_pair_features
        WHERE zone_type='{zt}' AND year={year}
        GROUP BY 1,2
        """
        ).show()


#Calculamos el parámetro k
#Se elabora un cálculo que elige k para que el error cuadrático total sea mínimo


In [203]:
con.sql(f"""SELECT id_{zt}_ine,
                id_{zt}_mitma
          FROM silver.ine_mitma_zones
        
        """)

┌──────────────────┬────────────────────┐
│ id_districts_ine │ id_districts_mitma │
│      int32       │       int32        │
├──────────────────┼────────────────────┤
│                5 │                  1 │
│                5 │                  1 │
│               12 │                 10 │
│               12 │                 10 │
│               12 │                 10 │
│               12 │                 10 │
│               12 │                 10 │
│               12 │                 10 │
│               12 │                 10 │
│               21 │                194 │
│                · │                 ·  │
│                · │                 ·  │
│                · │                 ·  │
│            17002 │              17009 │
│            17014 │              17010 │
│            17014 │              17010 │
│            17018 │              16501 │
│            17021 │              17024 │
│            17027 │              17265 │
│            17030 │              

In [ ]:
con.sql(f"""
        WITH aux_dim AS(
                SELECT *
                FROM silver.ine_mitma_zones
        )
        
        
        
        SELECT id_districts_mitma, SUM(population) as total_pop
        FROM silver.spain_population p 
        LEFT JOIN aux_dim a 
        ON p.id_zone = a.id_sections_ine 
        WHERE year = 2023
        AND id_districts_mitma NOT NULL
        GROUP BY id_districts_mitma
        ORDER BY total_pop DESC
        """)

┌────────────────────┬───────────┐
│ id_districts_mitma │ total_pop │
│       int32        │  double   │
├────────────────────┼───────────┤
│               7226 │  239775.0 │
│              33812 │  224658.0 │
│              33610 │  214994.0 │
│              34088 │  211338.0 │
│               8110 │  210314.0 │
│              33337 │  203315.0 │
│              34357 │  196039.0 │
│               7401 │  163899.0 │
│              34530 │  157650.0 │
│               7769 │  153263.0 │
│                 ·  │       ·   │
│                 ·  │       ·   │
│                 ·  │       ·   │
│              56906 │    1343.0 │
│              36838 │    1313.0 │
│              21239 │    1313.0 │
│              10619 │    1256.0 │
│              25869 │    1204.0 │
│              10754 │    1073.0 │
│               5911 │    1052.0 │
│              16998 │    1018.0 │
│              24163 │     671.0 │
│              45690 │     548.0 │
├────────────────────┴───────────┤
│ 3743 rows (20 show

In [260]:
zt = "districts"
year = 2023
query = f"""--sql
   
        WITH
        od AS (SELECT
                TRY_CAST(id_origin AS INTEGER) AS id_origin,
                TRY_CAST(id_destination AS INTEGER) AS id_destination,
                round(SUM(n_trips),1) AS actual_trips
            FROM silver.od_trips
            WHERE zone_type = 'distritcs'
              AND YEAR(date) = {year}
              AND id_origin <> id_destination
            GROUP BY 1,2 ),
       
        dist AS (
          SELECT
                zone_type,
                id_origin AS a,
                id_destination AS b,
                distance_km
            FROM silver.zone_pairs
            WHERE zone_type = 'districts'),
        aux_inc AS(
          SELECT DISTINCT id_districts_ine,
                id_{zt}_mitma
          FROM silver.ine_mitma_zones
          ),
        inc AS (
            SELECT 
              a.id_districts_mitma as id_zone,
              SUM(rent) AS inc
            FROM silver.average_rent i
            LEFT JOIN aux_inc a 
            ON i.id_zone = a.id_districts_ine 
            WHERE i.year = {year}
            AND id_{zt}_mitma NOT NULL
            GROUP BY id_{zt}_mitma
          ),
        aux_pop AS(
          SELECT DISTINCT id_sections_ine,
                id_{zt}_mitma
          FROM silver.ine_mitma_zones
        ),

      pop AS (
        SELECT id_districts_mitma as id_zone, 
          SUM(population) as population
        FROM silver.spain_population p 
        LEFT JOIN aux_pop a 
        ON p.id_zone = a.id_sections_ine 
        WHERE year = {year}
        AND id_districts_mitma NOT NULL
        GROUP BY id_{zt}_mitma
        ),
      joined AS(
        SELECT
                '{zt}' AS zone_type,
                {year} AS year,
                o.id_origin,
                o.id_destination,
                d.distance_km,
                o.actual_trips,
                po.population AS pop_origin,
                rd.inc AS inc_destination
            FROM od o
            LEFT JOIN dist d
              ON LEAST(o.id_origin, o.id_destination) = d.a
             AND GREATEST(o.id_origin, o.id_destination) = d.b
            LEFT JOIN pop po ON o.id_origin = po.id_zone
            LEFT JOIN inc rd ON o.id_destination = rd.id_zone)
      SELECT

            zone_type,
            year,
            id_origin,
            id_destination,
            distance_km,
            actual_trips,
            pop_origin,
            inc_destination,
            ROUND((pop_origin * inc_destination)/ NULLIF(POWER(GREATEST(distance_km, {0.1}), {2}), 0),3) AS x_ij,
            CURRENT_TIMESTAMP AS ingestion_date
        FROM joined

            """

con.sql(f""" MERGE INTO gold.gravity_pair_features as target
          USING ({query} )AS source
                    ON target.zone_type = source.zone_type 
                    AND target.year = source.year
                    AND target.id_origin = source.id_origin
                    WHEN NOT MATCHED THEN
                        INSERT BY NAME;
        
         """)


In [262]:
con.sql("SELECT * FROM gold.gravity_pair_features ORDER BY x_ij DESC")

┌───────────┬───────┬───────────┬────────────────┬─────────────┬──────────────┬────────────┬─────────────────┬────────────────┬────────────────────────────┐
│ zone_type │ year  │ id_origin │ id_destination │ distance_km │ actual_trips │ pop_origin │ inc_destination │      x_ij      │       ingestion_date       │
│  varchar  │ int32 │   int32   │     int32      │   double    │    double    │   double   │     double      │     double     │         timestamp          │
├───────────┼───────┼───────────┼────────────────┼─────────────┼──────────────┼────────────┼─────────────────┼────────────────┼────────────────────────────┤
│ districts │  2023 │     32367 │          32515 │       0.242 │      55879.1 │    27265.0 │         18664.0 │ 8689194044.123 │ 2026-01-04 22:35:03.440033 │
│ districts │  2023 │      9157 │           8768 │       1.269 │      89858.1 │    35146.0 │        225625.0 │ 4924247575.544 │ 2026-01-04 22:35:03.440033 │
│ districts │  2023 │      8110 │          10381 │        

In [264]:
con.sql(
        f"""
        SELECT
          zone_type, year,
          COUNT(*) AS n_pairs,
          ROUND(SUM(actual_trips),2) AS total_actual_trips,
          ROUND(AVG(distance_km),2) AS avg_distance_km
        FROM gold.gravity_pair_features
        WHERE zone_type='{zt}' AND year={year}
        GROUP BY 1,2
        """
    ).show()

┌───────────┬───────┬─────────┬────────────────────┬─────────────────┐
│ zone_type │ year  │ n_pairs │ total_actual_trips │ avg_distance_km │
│  varchar  │ int32 │  int64  │       double       │     double      │
├───────────┼───────┼─────────┼────────────────────┼─────────────────┤
│ districts │  2023 │ 1668920 │        693415069.8 │          206.65 │
└───────────┴───────┴─────────┴────────────────────┴─────────────────┘



In [268]:
build_gravity_pair_features(
    con,
    year = 2023,
    zone_type = "districts")

[BQ2] gold.gravity_pair_features built for zone_type=districts, year=2023.
┌───────────┬───────┬─────────┬────────────────────┬─────────────────┐
│ zone_type │ year  │ n_pairs │ total_actual_trips │ avg_distance_km │
│  varchar  │ int32 │  int64  │       double       │     double      │
├───────────┼───────┼─────────┼────────────────────┼─────────────────┤
│ districts │  2023 │ 1668920 │        693415069.8 │          206.65 │
└───────────┴───────┴─────────┴────────────────────┴─────────────────┘



In [269]:
def fit_gravity_k(
    con: duckdb.DuckDBPyConnection,
    year: int = 2023,
    zone_type: str = "districts",
) -> None:
    _ensure_gold_schema(con)
    zt = _normalize_zone_type(zone_type)

    con.sql("""
        CREATE TABLE IF NOT EXISTS gold.gravity_params (
            zone_type VARCHAR,
            year INTEGER,
            k DOUBLE,
            n_pairs_used BIGINT,
            fitted_at TIMESTAMP
        );
    """)
    con.sql(f"DELETE FROM gold.gravity_params WHERE zone_type='{zt}' AND year={year};")

    con.sql(
        f"""
        INSERT INTO gold.gravity_params
        SELECT
            '{zt}' AS zone_type,
            {year} AS year,
            SUM(x_ij * actual_trips) / NULLIF(SUM(x_ij * x_ij), 0) AS k,
            COUNT(*) AS n_pairs_used,
            CURRENT_TIMESTAMP AS fitted_at
        FROM gold.gravity_pair_features
        WHERE zone_type='{zt}' AND year={year}
          AND x_ij IS NOT NULL AND x_ij > 0
          AND actual_trips IS NOT NULL
        ;
        """
    )

    print(f"[BQ2] gold.gravity_params fitted for zone_type={zt}, year={year}.")
    con.sql(f"SELECT * FROM gold.gravity_params WHERE zone_type='{zt}' AND year={year};").show()


In [270]:
fit_gravity_k(
    con,
    year= 2023,
    zone_type= "districts",
)

[BQ2] gold.gravity_params fitted for zone_type=districts, year=2023.
┌───────────┬───────┬────────────────────────┬──────────────┬───────────────────────────┐
│ zone_type │ year  │           k            │ n_pairs_used │         fitted_at         │
│  varchar  │ int32 │         double         │    int64     │         timestamp         │
├───────────┼───────┼────────────────────────┼──────────────┼───────────────────────────┤
│ districts │  2023 │ 0.00011595202851462751 │      1565051 │ 2026-01-04 22:47:24.83954 │
└───────────┴───────┴────────────────────────┴──────────────┴───────────────────────────┘



In [274]:
#TABLA PRINCIPAL
def build_infrastructure_gaps(
    con: duckdb.DuckDBPyConnection,
    year: int = 2023,
    zone_type: str = "districts",
) -> None:
    _ensure_gold_schema(con)
    zt = _normalize_zone_type(zone_type)

    con.sql("""
        CREATE TABLE IF NOT EXISTS gold.infrastructure_gaps (
            zone_type VARCHAR,
            year INTEGER,
            id_origin INTEGER,
            id_destination INTEGER,
            distance_km DOUBLE,
            actual_trips DOUBLE,
            theoretical_trips DOUBLE,
            mismatch_ratio DOUBLE,
            gap DOUBLE,
            ingestion_date TIMESTAMP
        );
    """)

    con.sql(f"DELETE FROM gold.infrastructure_gaps WHERE zone_type='{zt}' AND year={year};")

    con.sql(
        f"""
        INSERT INTO gold.infrastructure_gaps
        WITH kpar AS (
            SELECT k
            FROM gold.gravity_params
            WHERE zone_type='{zt}' AND year={year}
        )
        SELECT
            f.zone_type,
            f.year,
            f.id_origin,
            f.id_destination,
            f.distance_km,
            f.actual_trips,
            (k.k * f.x_ij) AS theoretical_trips,
            f.actual_trips / NULLIF((k.k * f.x_ij), 0) AS mismatch_ratio,
            GREATEST((k.k * f.x_ij) - f.actual_trips, 0) AS gap,
            CURRENT_TIMESTAMP AS ingestion_date
        FROM gold.gravity_pair_features f
        CROSS JOIN kpar k
        WHERE f.zone_type='{zt}' AND f.year={year}
          AND f.x_ij IS NOT NULL AND f.x_ij > 0
        ;
        """
    )

    print(f"[BQ2] gold.infrastructure_gaps built for zone_type={zt}, year={year}.")
    print("[BQ2] Top potential gaps (lowest mismatch_ratio, with some volume):")
    con.sql(
        f"""
        SELECT *
        FROM gold.infrastructure_gaps
        WHERE zone_type='{zt}' AND year={year}
          AND actual_trips >= 50
        ORDER BY mismatch_ratio ASC
        LIMIT 10
        """
    ).show()

In [275]:

# Zone ranking - tabla opcional que clasifica las zonas en función del nivel/calidad de servicio - usando los gaps en las infra
def create_zone_gap_ranking_view(con: duckdb.DuckDBPyConnection) -> None:
    _ensure_gold_schema(con)

    con.sql("""
        CREATE OR REPLACE VIEW gold.zone_gap_ranking AS
        WITH
        out_gap AS (
            SELECT zone_type, year, id_origin AS id_zone, SUM(gap) AS gap_outgoing
            FROM gold.infrastructure_gaps
            GROUP BY 1,2,3
        ),
        in_gap AS (
            SELECT zone_type, year, id_destination AS id_zone, SUM(gap) AS gap_incoming
            FROM gold.infrastructure_gaps
            GROUP BY 1,2,3
        ),
        merged AS (
            SELECT
                COALESCE(o.zone_type, i.zone_type) AS zone_type,
                COALESCE(o.year, i.year) AS year,
                COALESCE(o.id_zone, i.id_zone) AS id_zone,
                COALESCE(o.gap_outgoing, 0) AS gap_outgoing,
                COALESCE(i.gap_incoming, 0) AS gap_incoming
            FROM out_gap o
            FULL JOIN in_gap i
              ON o.zone_type=i.zone_type AND o.year=i.year AND o.id_zone=i.id_zone
        ),
        attrs AS (
            SELECT
                dz.zone_type,
                p.year,
                dz.id_zone,
                COALESCE(p.population, (SELECT AVG(population) FROM silver.spain_population WHERE year=p.year)) AS population,
                COALESCE(r.rent, (SELECT AVG(rent) FROM silver.average_rent WHERE year=p.year)) AS rent
            FROM silver.dim_zones dz
            LEFT JOIN silver.spain_population p
              ON p.id_zone = dz.id_zone
            LEFT JOIN silver.average_rent r
              ON r.id_zone = dz.id_zone AND r.year = p.year
        )
        SELECT
            m.zone_type,
            m.year,
            m.id_zone,
            m.gap_outgoing,
            m.gap_incoming,
            (m.gap_outgoing + m.gap_incoming) AS gap_total,
            a.population,
            a.rent,
            (m.gap_outgoing + m.gap_incoming) * COALESCE(a.population, 1) * (COALESCE(a.rent, 1) / 1000.0) AS weighted_gap,
            RANK() OVER (
                PARTITION BY m.zone_type, m.year
                ORDER BY (m.gap_outgoing + m.gap_incoming) * COALESCE(a.population, 1) * (COALESCE(a.rent, 1) / 1000.0) DESC
            ) AS rank_worst_served
        FROM merged m
        LEFT JOIN attrs a
          ON a.zone_type = m.zone_type AND a.year = m.year AND a.id_zone = m.id_zone
        ;
    """)

    print("[BQ2] gold.zone_gap_ranking VIEW created/updated.")


In [276]:
def run_bq2(
    con: duckdb.DuckDBPyConnection,
    year: int = 2023,
    zone_types: list[str] | None = None,
    dist_floor_km: float = 0.1,
    dist_power: int = 2,
) -> None:
    _ensure_gold_schema(con)

    if zone_types is None:
        zone_types = ["districts", "municiples", "gaus"]

    zone_types_norm = [_normalize_zone_type(z) for z in zone_types]

    for zt in zone_types_norm:
        build_gravity_pair_features(
            con=con,
            year=year,
            zone_type=zt,
            dist_floor_km=dist_floor_km,
            dist_power=dist_power,
            keep_only_with_distance=True,
        )
        fit_gravity_k(con=con, year=year, zone_type=zt)
        build_infrastructure_gaps(con=con, year=year, zone_type=zt)

    create_zone_gap_ranking_view(con)

In [277]:
run_bq2(
    con,
    year = 2023, zone_types=["districts"])

[BQ2] gold.gravity_pair_features built for zone_type=districts, year=2023.
┌───────────┬───────┬─────────┬────────────────────┬─────────────────┐
│ zone_type │ year  │ n_pairs │ total_actual_trips │ avg_distance_km │
│  varchar  │ int32 │  int64  │       double       │     double      │
├───────────┼───────┼─────────┼────────────────────┼─────────────────┤
│ districts │  2023 │ 1668920 │        693415069.8 │          206.65 │
└───────────┴───────┴─────────┴────────────────────┴─────────────────┘

[BQ2] gold.gravity_params fitted for zone_type=districts, year=2023.
┌───────────┬───────┬────────────────────────┬──────────────┬────────────────────────────┐
│ zone_type │ year  │           k            │ n_pairs_used │         fitted_at          │
│  varchar  │ int32 │         double         │    int64     │         timestamp          │
├───────────┼───────┼────────────────────────┼──────────────┼────────────────────────────┤
│ districts │  2023 │ 0.00011595202851462748 │      1565051 │ 202